# Half-system mixed-mode densities across sizes

Compare $20\times24$, $20\times28$, and $20\times32$ hard-wall pure-state endpoint ensembles at $\alpha_1=1$, $\alpha_2=30$, and overcomplete Wannier (OW) range $n_{\mathrm{shell}}=1$. Each size has 100 independent Born trajectories at cycle $2N_y$.

## CPU allocation
Edit the inclusive CPU range below. Calculations use CPU linear algebra on saved frames; no dynamics are rerun.

In [ ]:
import os
# Preserve the original kernel affinity so this cell can be rerun with a new range.
if "AVAILABLE_CPUS" not in globals():
    AVAILABLE_CPUS = sorted(os.sched_getaffinity(0))
CPU_RANGE = (AVAILABLE_CPUS[0], AVAILABLE_CPUS[min(3, len(AVAILABLE_CPUS)-1)])
selected_cpus = list(range(CPU_RANGE[0], CPU_RANGE[1] + 1))
if not selected_cpus or not set(selected_cpus).issubset(AVAILABLE_CPUS):
    raise ValueError("CPU_RANGE must be an inclusive range of available CPUs")
os.sched_setaffinity(0, selected_cpus)
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = str(len(selected_cpus))
from threadpoolctl import threadpool_limits
blas_limit = threadpool_limits(limits=len(selected_cpus))
print("Allocated CPUs:", selected_cpus)

## Estimator and normalization

For each trajectory select all $x$, $y=0,\ldots,N_y/2-1$, and both orbitals using $i=2N_xy+2x+\mu$. This gives 480, 560, or 640 subsystem modes. Let $C_A=F_AF_A^\dagger$ and use the engine's centered convention $G_A=2C_A-I$ (the paper's $G$ denotes the uncentered correlation matrix). Diagonalize each reduced matrix before pooling eigenvalues at the same size.

Retain $|\lambda|<1-10^{-8}$ to exclude numerically unresolved endpoint modes. Normalize each size **separately** by its pooled retained-mode count and bin width. Each conditional density integrates to one; trajectories with more retained modes contribute more entries.

All three curves share 100 equal-width bins on $[-1,1]$ and a logarithmic vertical axis. Empty bins are gaps, with no pseudocounts. Endpoints have equal scaled depth $2N_y$, not equal absolute cycle count. No translated cuts are averaged.

## Configuration and provenance
Verify completion receipts, NPZ checksums, sample IDs, executed source identities, and configuration.

In [ ]:
from pathlib import Path
import hashlib, json, time, platform
from datetime import datetime, timezone
import numpy as np
import scipy
from scipy.linalg import eigvalsh
from tqdm.auto import tqdm
from IPython.display import display, Markdown, Image
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p/"PROJECT_ADMIN/REPO_POLICY.md").is_file())
BUNDLE = REPO_ROOT/"00_WORKSPACE/CURRENT/final_production_new_designs/09_pure_tangent_replay_acquisition"
REVISION = "pure_tangent_replay_nx20_ny24-32_hard-soft_a1-1-3_s100_v1"
OUTPUT_DIR = BUNDLE/"analysis_outputs/half_system_centered_spectrum_n20_sizes_hard_alpha1_v1"
EXPECTED_CONFIG_HASH = "d1a7c212d8f2c06b0f774af14272a2c71b9d0482240f0c4c43be75a9cd494be4"
NX, SAMPLE_COUNT = 20, 100
NY_VALUES = [24, 28, 32]
BOUND_TOL, HERMITICITY_TOL, TRACE_TOL = 1e-8, 1e-12, 1e-9
FORCE_RECOMPUTE = False
OUTPUT_DIR.mkdir(exist_ok=True)
def require(condition, message):
    if not condition:
        raise ValueError(message)
def sha256(path):
    with Path(path).open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()
print("Campaign:", REVISION)
print("Hard wall, alpha_1=1, alpha_2=30, nshell=1, pure half-filled initialization,")
print("raster-y, perfect correction, no postselection; 100 samples per size, cycles 2Ny.")
print("Output:", OUTPUT_DIR)

## Sample-resolved spectra
Process one frame batch at a time. Check orthonormality, Hermiticity, finite values, bounds, trace closure, and an alternate eigensolver on sample 0 at each size. Cache reuse requires matching identities and checksums.

In [ ]:
def analyze_size(NY):
    AY = NY//2
    MODES = 2*NX*AY
    expected_batch_size = 25 if NY == 32 else 50
    expected_batches = SAMPLE_COUNT//expected_batch_size
    DATA_DIR = BUNDLE/"gpu_data"/REVISION/f"hard/Ny{NY:03d}/alpha1_1"
    CACHE = OUTPUT_DIR/f"centered_spectra_Ny{NY:03d}.npz"
    DIAGNOSTICS = OUTPUT_DIR/f"spectra_diagnostics_Ny{NY:03d}.json"
    indices = np.array([2*NX*y+2*x+mu for y in range(AY) for x in range(NX) for mu in range(2)])
    require(MODES == 20*NY and np.array_equal(indices, np.arange(MODES)), "Subsystem index mismatch")
    receipts = sorted(DATA_DIR.glob("*.complete.json"))
    require(len(receipts) == expected_batches, "Unexpected completion count")
    require(len(list(DATA_DIR.glob("*.npz"))) == expected_batches, "Unexpected result count")
    inputs, ids, source_hashes = [], [], None
    for receipt in tqdm(receipts, desc=f"Ny={NY}: verify checksums", unit="batch"):
        r = json.loads(receipt.read_text())
        for key, value in {"Nx": NX, "Ny": NY, "cycles": 2*NY, "construction": "hard",
                           "alpha_1": 1.0, "alpha_2": 30.0, "nshell": 1,
                           "configuration_sha256": EXPECTED_CONFIG_HASH,
                           "sampling_revision": REVISION, "status": "complete",
                           "canonical_entry_point": "classA_U1FGTN_gpu.run_markov_circuit"}.items():
            require(r.get(key) == value, f"Receipt mismatch: {receipt.name}: {key}")
        filename = r["result_filename"]
        require(Path(filename).name == filename, "Result filename must be local to data directory")
        result = DATA_DIR / filename
        require(receipt.name == result.stem + ".complete.json", "Receipt filename mismatch")
        digest = sha256(result)
        require(result.stat().st_size == r["result_bytes"], "Result byte count mismatch")
        require(digest == r["result_sha256"], "Result SHA-256 mismatch")
        require(bool(r["source_hashes"]), "Missing source identities")
        if source_hashes is None:
            source_hashes = r["source_hashes"]
        require(r["source_hashes"] == source_hashes, "Inconsistent executed source identities")
        with np.load(result, allow_pickle=False) as z:
            for key, value in {"Nx": NX, "Ny": NY, "cycles_total": 2*NY,
                               "construction": "hard", "alpha_1": 1.0, "alpha_2": 30.0,
                               "nshell": 1, "dtype": "complex128", "sequence": "raster_y",
                               "configuration_sha256": EXPECTED_CONFIG_HASH,
                               "sampling_revision": REVISION,
                               "canonical_entry_point": r["canonical_entry_point"],
                               "task_id": r["task_id"], "batch_seed": r["seed"]}.items():
                require(z[key].item() == value, f"NPZ metadata mismatch: {key}")
            sample_ids = z["case_sample_indices"].tolist()
            require(sample_ids == r["case_sample_indices"] and len(sample_ids) == expected_batch_size,
                    "Sample identity mismatch")
            require(z["global_sample_indices"].tolist() == r["global_sample_indices"],
                    "Global sample identity mismatch")
            require(json.loads(z["source_hashes_json"].item()) == source_hashes,
                    "NPZ source identity mismatch")
        ids.extend(sample_ids)
        inputs.append({"result": str(result.relative_to(REPO_ROOT)), "sha256": digest,
                       "bytes": r["result_bytes"], "receipt_sha256": sha256(receipt),
                       "receipt": str(receipt.relative_to(REPO_ROOT)),
                       "sample_ids": sample_ids, "seed": r["seed"], "task_id": r["task_id"]})
    require(sorted(ids) == list(range(SAMPLE_COUNT)), "Missing or duplicate sample IDs")
    contract = {"schema": "half_system_centered_spectrum_v1", "inputs": inputs,
                "configuration_sha256": EXPECTED_CONFIG_HASH, "source_hashes": source_hashes,
                "subsystem_indices": indices.tolist(), "formula": "G_A=2*F_A@F_A.conj().T-I",
                "bounds_tolerance": BOUND_TOL, "hermiticity_tolerance": HERMITICITY_TOL,
                "trace_absolute_tolerance": TRACE_TOL}
    identity = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()
    if NY == 32 and not CACHE.exists() and not DIAGNOSTICS.exists() and not FORCE_RECOMPUTE:
        prior = BUNDLE/"analysis_outputs/half_system_centered_spectrum_n20x32_hard_alpha1_v1"
        if (prior/"centered_spectra.npz").is_file() and (prior/"diagnostics.json").is_file():
            prior_diagnostics = json.loads((prior/"diagnostics.json").read_text())
            if (prior_diagnostics.get("analysis_identity") == identity
                    and prior_diagnostics.get("spectra_sha256") == sha256(prior/"centered_spectra.npz")):
                import shutil
                shutil.copyfile(prior/"centered_spectra.npz", CACHE)
                shutil.copyfile(prior/"diagnostics.json", DIAGNOSTICS)
                print("Reusing verified Ny=32 spectra from the previous analysis.")
    start = time.perf_counter()
    reuse = False
    if CACHE.exists() and DIAGNOSTICS.exists() and not FORCE_RECOMPUTE:
        previous = json.loads(DIAGNOSTICS.read_text())
        reuse = (previous.get("analysis_identity") == identity
                 and previous.get("spectra_sha256") == sha256(CACHE))
    if reuse:
        with np.load(CACHE, allow_pickle=False) as z:
            require(z["analysis_identity"].item() == identity, "Cached identity mismatch")
            require(np.array_equal(z["sample_ids"], np.arange(SAMPLE_COUNT)), "Cached sample mismatch")
            spectra = z["eigenvalues"]
        diagnostics = previous
        print("Loaded verified spectra cache; no eigendecompositions repeated.")
    else:
        spectra = np.empty((SAMPLE_COUNT, MODES), dtype=np.float64)
        sample_checks = []
        crosscheck = None
        progress = tqdm(total=SAMPLE_COUNT, desc=f"Ny={NY}: eigenspectra", unit="trajectory")
        try:
            for info in inputs:
                with np.load(REPO_ROOT / info["result"], allow_pickle=False) as z:
                    frames, ranks = z["final_frame"], z["final_ranks"]
                require(frames.dtype == np.complex128 and frames.shape[:2] == (expected_batch_size, 2*NX*NY),
                        "Unexpected final-frame shape or dtype")
                require(ranks.shape == (expected_batch_size,) and np.issubdtype(ranks.dtype, np.integer),
                        "Invalid ranks array")
                for row, sample_id in enumerate(info["sample_ids"]):
                    rank = int(ranks[row])
                    require(0 <= rank <= frames.shape[2], "Invalid active-column rank")
                    F = frames[row, :, :rank]
                    require(np.isfinite(F).all(), f"Nonfinite frame: {sample_id}")
                    gram_residual = float(np.max(np.abs(F.conj().T @ F - np.eye(rank)))) if rank else 0.0
                    require(gram_residual <= BOUND_TOL, f"Nonorthonormal frame: {sample_id}")
                    FA = F[indices, :]
                    C = FA @ FA.conj().T
                    G = 2*C - np.eye(MODES)
                    require(np.isfinite(G).all(), f"Nonfinite covariance: {sample_id}")
                    herm = float(np.max(np.abs(G-G.conj().T)))
                    require(herm <= HERMITICITY_TOL, f"Non-Hermitian covariance: {sample_id}")
                    values = eigvalsh((G+G.conj().T)/2, check_finite=True, driver="evr")
                    require(values.shape == (MODES,) and np.isfinite(values).all(), "Invalid spectrum")
                    require(values.min() >= -1-BOUND_TOL and values.max() <= 1+BOUND_TOL,
                            f"Spectrum outside numerical tolerance: {sample_id}")
                    trace_error = float(abs(values.sum()-np.trace(G)))
                    require(trace_error <= TRACE_TOL, f"Trace closure failed: {sample_id}")
                    if sample_id == 0:
                        alternate = 2*eigvalsh(C, driver="evd")-1
                        crosscheck = float(np.max(np.abs(values-alternate)))
                        require(crosscheck <= 1e-10, "Occupation-spectrum cross-check failed")
                    spectra[sample_id] = values
                    sample_checks.append({"sample_id": sample_id, "rank": rank,
                                          "gram_residual": gram_residual, "hermiticity_residual": herm,
                                          "trace_error": trace_error, "raw_min": float(values.min()),
                                          "raw_max": float(values.max())})
                    progress.update()
                del frames, ranks, F, FA, C, G
        finally:
            progress.close()
        temporary = CACHE.with_suffix(".tmp.npz")
        np.savez_compressed(temporary, eigenvalues=spectra, sample_ids=np.arange(SAMPLE_COUNT),
                            subsystem_indices=indices, analysis_identity=np.array(identity))
        temporary.replace(CACHE)
        diagnostics = {"analysis_identity": identity, "contract": contract,
                       "created_utc": datetime.now(timezone.utc).isoformat(),
                       "spectra_sha256": sha256(CACHE), "sample_checks": sample_checks,
                       "sample0_crosscheck_max_error": crosscheck,
                       "calculation_seconds": time.perf_counter()-start,
                       "runtime": {"python": platform.python_version(), "numpy": np.__version__,
                                   "scipy": scipy.__version__, "cpus": selected_cpus},
                       "independent_sampling_unit": "whole Born trajectory",
                       "canonical_dynamics_entry_point": "classA_U1FGTN_gpu.run_markov_circuit",
                       "dynamics_rerun": False}
        DIAGNOSTICS.write_text(json.dumps(diagnostics, indent=2, allow_nan=False)+"\n")
    require(spectra.shape == (SAMPLE_COUNT, MODES) and np.isfinite(spectra).all(), "Invalid cached spectra")
    print("Spectrum array:", spectra.shape, "raw extrema:", spectra.min(), spectra.max())
    return spectra, diagnostics

spectra_by_size, checks_by_size = {}, {}
for ny in NY_VALUES:
    spectra_by_size[ny], checks_by_size[ny] = analyze_size(ny)

## Mixed-mode density overlay
Edit MIXED_TOL and BIN_COUNT here, then rerun this plot cell and the diagnostics cell to rebin without repeating eigendecompositions.

In [ ]:
MIXED_TOL = 1e-8
BIN_COUNT = 100
require(0 < MIXED_TOL < 1, "Invalid endpoint tolerance")
require(isinstance(BIN_COUNT, int) and BIN_COUNT > 0, "Invalid bin count")
import matplotlib as mpl
import matplotlib.pyplot as plt
import subprocess
os.environ["TEXINPUTS"] = str(OUTPUT_DIR/"latex_support")+os.pathsep+os.environ.get("TEXINPUTS", "")
edges = np.linspace(-1, 1, BIN_COUNT+1)
widths, centers = np.diff(edges), (edges[:-1]+edges[1:])/2
density_by_size, csv_rows, summaries = {}, [], {}
for ny, spectra in spectra_by_size.items():
    require(spectra.shape == (100, 20*ny) and np.isfinite(spectra).all(), "Invalid spectra")
    require(spectra.min() >= -1-BOUND_TOL and spectra.max() <= 1+BOUND_TOL, "Invalid bounds")
    mask = np.abs(spectra) < 1-MIXED_TOL
    values = spectra[mask]
    require(values.size > 0, f"No mixed modes at Ny={ny}")
    counts, _ = np.histogram(values, bins=edges)
    density = counts/(values.size*widths)
    area = float(np.sum(density*widths))
    require(int(counts.sum()) == values.size and abs(area-1) < 1e-14, "Normalization failed")
    density_by_size[ny] = np.where(counts > 0, density, np.nan)
    for left, right, count, value in zip(edges[:-1], edges[1:], counts, density):
        csv_rows.append([ny, left, right, count, value])
    checks = checks_by_size[ny]
    summaries[str(ny)] = {
        "Nx":NX, "Ny":ny, "cycle":2*ny, "samples":100, "modes_per_sample":20*ny,
        "total_modes":int(spectra.size), "retained_modes":int(values.size),
        "excluded_modes":int(spectra.size-values.size),
        "retained_fraction":float(values.size/spectra.size),
        "mean_retained_per_sample":float(mask.sum(axis=1).mean()),
        "retained_per_sample":mask.sum(axis=1).tolist(), "density_integral":area,
        "raw_minimum":float(spectra.min()), "raw_maximum":float(spectra.max()),
        "max_hermiticity_residual":max(x["hermiticity_residual"] for x in checks["sample_checks"]),
        "max_trace_error":max(x["trace_error"] for x in checks["sample_checks"]),
        "sample0_crosscheck_max_error":checks["sample0_crosscheck_max_error"],
        "analysis_identity":checks["analysis_identity"],
        "spectra_sha256":sha256(OUTPUT_DIR/f"centered_spectra_Ny{ny:03d}.npz"),
    }
np.savetxt(OUTPUT_DIR/"mixed_mode_density_overlay.csv", csv_rows, delimiter=",",
           header="Ny,left_edge,right_edge,count,conditional_probability_density", comments="",
           fmt=["%d","%.17g","%.17g","%d","%.17g"])
mpl.rcParams.update({
    "font.family":"sans-serif","font.sans-serif":["CMU Sans Serif"],"font.size":8,
    "axes.labelsize":8,"axes.titlesize":8,"xtick.labelsize":8,"ytick.labelsize":8,
    "text.usetex":True,"pdf.fonttype":42,"xtick.direction":"in","ytick.direction":"in"})
fig, ax = plt.subplots(figsize=(3.375, 2.85))
styles = [("#c62828","^",":"),("#2e7d32","s","--"),("#1565c0","o","-")]
for ny, (color, marker, linestyle) in zip(NY_VALUES, styles):
    values = density_by_size[ny]
    ax.stairs(values, edges, color=color, linestyle=linestyle, linewidth=0.85,
              label=rf"$20\times{ny}$")
    ax.plot(centers[::8], values[::8], linestyle="none", marker=marker,
            markersize=2.6, markerfacecolor="none", color=color)
ax.set_yscale("log")
ax.set(xlim=(-1,1), xlabel=r"Centered eigenvalue $\lambda$",
       ylabel=r"Conditional density $\rho_{\mathrm{mixed}}(\lambda)$",
       title=r"Half-system mixed modes: hard wall, $\alpha_1=1$")
ax.set_xticks([-1,-0.5,0,0.5,1])
ax.tick_params(top=True,right=True)
ax.text(0.5,0.96,r"$S=100$ per size; cycle $2N_y$"
        +"\n"+f"Endpoint tolerance {MIXED_TOL:.0e}; unit area per size",
        transform=ax.transAxes,ha="center",va="top",fontsize=8)
ax.legend(frameon=False,loc="upper center",bbox_to_anchor=(0.5,0.80),
          ncol=3,columnspacing=0.6,handlelength=1.5,handletextpad=0.4,fontsize=8)
fig.tight_layout(pad=0.5)
fig.savefig(OUTPUT_DIR/"mixed_mode_density_overlay.pdf")
plt.close(fig)
subprocess.run(["pdftoppm","-r","300","-singlefile","-png",
                str(OUTPUT_DIR/"mixed_mode_density_overlay.pdf"),
                str(OUTPUT_DIR/"mixed_mode_density_overlay")],check=True)
display(Image(filename=str(OUTPUT_DIR/"mixed_mode_density_overlay.png"),width=550))
caption = (
    "Half-system centered-covariance mixed-mode densities for hard-wall pure-state "
    "ensembles at Nx=20, Ny=24,28,32, alpha_1=1, alpha_2=30, nshell=1, "
    "100 independent Born trajectories per size. Pure half-filled initialization, "
    "raster-y, perfect correction, no postselection. Endpoints are cycles 48,56,64 "
    "(2Ny), respectively. Subsystem: all x, y=0..Ny/2-1, both orbitals. "
    "Each sample is diagonalized before pooling by size. "
    f"Retain abs(lambda)<1-{MIXED_TOL:g} and use {BIN_COUNT} common bins on [-1,1]. "
    "Each curve is normalized by its own retained mode count and bin width; "
    "bin-integrated area is one. Logarithmic vertical axis; empty bins are gaps. "
    "No uncertainty intervals or fits are shown.")
(OUTPUT_DIR/"figure_caption.txt").write_text(caption+"\n")
display(Markdown(caption))

## Numerical diagnostics and completion
Each size must have 100 samples and unit integrated conditional density. Raw caches preserve all eigenvalues before endpoint exclusion.

In [ ]:
summary = {
    "schema":"half_system_mixed_mode_size_overlay_v1",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "endpoint_tolerance":MIXED_TOL,"bin_count":BIN_COUNT,
    "normalization":"separate unit-area pooled retained-mode density per size",
    "cut":"all x, y=0..Ny/2-1, both orbitals",
    "sizes":summaries,"all_checks_passed":True,
}
summary["outputs"] = {p.name:sha256(p) for p in OUTPUT_DIR.iterdir()
                     if p.suffix in (".npz",".csv",".pdf",".png") or p.name=="figure_caption.txt"}
(OUTPUT_DIR/"overlay_diagnostics.json").write_text(json.dumps(summary,indent=2,allow_nan=False)+"\n")
for ny,row in summaries.items():
    print(f"20x{ny}: {row['retained_modes']}/{row['total_modes']} retained; "
          f"area={row['density_integral']:.16g}; "
          f"max trace error={row['max_trace_error']:.3e}")
print("Complete: overlay, per-size raw spectra and provenance, CSV, PDF, PNG:",OUTPUT_DIR)